# RHO-FFT · 第二次会话（Colab）：零阶更新到底有没有用，以及剂量响应实验

这次会话做两件事，都只用种子 42，不重复第一次会话已经跑过的运行（那 12 个运行的结果从仓库里的参考文件读入，配对比较）：

1. **按 HO-FL 作者自己的设置，检验他们关于零阶更新的表述**：
   - rebuttal 里的 Top-FO-Only（冻结底层，β=0.75）：AG News 上作者报告 86.69 → 83.37，我们第一次会话的同一实验是 85.99 → 85.95；SQuAD 上作者报告 F1 86.92 → 83.65，我们这次补跑；
   - 论文表 9 的"所有设备都用最高的零阶分界（第 24 层），β=0"：作者报告 63.22；我们复现它，并加上同一设置下冻结底层的对照，看零阶覆盖 24 层时有没有贡献；
   - 默认设置（β=0.75）再跑一次并加上"归因探针"：把每轮更新带来的损失下降拆成一阶部分和零阶部分的贡献。这次运行的训练与第一次会话完全相同，所以它同时检验两次会话的结果能不能配对；
   - 零阶估计器的数值检查（不训练）：HO-FL 的有限差分与 autograd 算出的精确方向导数是否一致，BF16 与 FP32 各一遍。
2. **剂量响应实验（反事实设计）**：一条 20 轮的轨迹，每一轮用同一批报告算出所有剂量 × 8 种聚合规则 × 3 种故障强度的结果。它回答 H2，也回答"一阶报告不足 3 份时，零阶报告当压舱是否比暂停或稀释更好"，这决定 RHO-FFT 的聚合规则。

另外两组在预算够时运行：SQuAD 上的 Top-FO-Only（约 1.5 小时，最贵），以及步长与训练长度的对照（约 1.2 小时）。按第一次会话的实测耗时，全部步骤合计约 4 小时，预算 5.5 小时。

**步骤一览**

| 步骤 | 内容 | GPU |
|---|---|---|
| A1 到 A3 | 会话、取代码、建环境 | 不用 |
| B | 预演：小模型把所有流程走一遍，约四分钟 | 不用 |
| C1、C2 | 环境记录与单元测试；黄金值与基线回归（I1、I2） | 很少 |
| C3 | 零阶检验，AG News：默认设置加归因探针；统一分界 24，零阶开和关 | 约 35 分钟 |
| C4 | 剂量响应：40 轮起点运行 + 20 轮反事实轨迹 | 约 15 分钟 |
| C5 | 零阶估计器的数值检查（不训练） | 约 10 分钟 |
| C6 | 零阶检验，SQuAD：Top-FO-Only（可选） | 约 1.5 小时 |
| C7 | 对照：`fo_mean` 步长 ×2、FedAvg 步长 ×2、A 和 C 各 480 轮（可选） | 约 1.2 小时 |
| D | 两份反馈报告与图 | 不用 |
| E | 汇总、下载结果包、释放 GPU | |

放不进剩余预算的运行会被跳过并在报告里列出。运行时类型选 GPU（默认 G4）；代码只从 GitHub 读取（`REPO_REF`），不使用 Google Drive。

In [ ]:
#@title Parameters { display-mode: "form" }
# The code is read from GitHub only.
REPO_URL = "https://github.com/GuangLun2000/RHO-FFT.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}
ASK_FOR_TOKEN = True  #@param {type:"boolean"}

# Total wall-clock budget of this notebook, in hours.
BUDGET_HOURS = 5.5  #@param {type:"number"}

# "venv": Python 3.11 with the pinned dependencies of pyproject.toml (recommended).
# "system": Colab's own Python and torch; only a fallback, versions then differ from the pins.
ENV_MODE = "venv"  #@param ["venv", "system"]
# PyTorch installed only if the pinned 2.6.0 (CUDA 12.4) cannot use the GPU. The G4 runtime is an RTX PRO 6000 of
# the Blackwell generation, which needs a CUDA 12.8 build (PyTorch 2.7 or later).
TORCH_FALLBACK = "2.7.1"  #@param {type:"string"}

# What to run. Session 1 already ran blocks 1-3 for seed 42; their results are read from references, not trained again.
MODEL = "qwen"  #@param ["qwen", "opt"]
SEED = 42  #@param {type:"integer"}
# Runs that may share the GPU (each run is its own process; lowered automatically if memory does not allow it).
JOBS = 2  #@param {type:"integer"}

# Stages, in the order they run.
RUN_TINY_DRY_RUN = True  #@param {type:"boolean"}
RUN_TESTS = True  #@param {type:"boolean"}
RUN_REGRESSION = True  #@param {type:"boolean"}
RUN_ZO_AG_NEWS = True  #@param {type:"boolean"}
RUN_DOSE_CF = True  #@param {type:"boolean"}
RUN_ZO_CHECK = True  #@param {type:"boolean"}
RUN_ZO_SQUAD = True  #@param {type:"boolean"}
RUN_CONTROLS = True  #@param {type:"boolean"}
# Run the experiments even when the regression check did not pass (not recommended).
FORCE = False  #@param {type:"boolean"}
# Download the result bundle through the browser after every stage and before the runtime is released.
DOWNLOAD_BUNDLE = True  #@param {type:"boolean"}
# Release the GPU runtime after the last cell, and after an error in any cell.
RELEASE_GPU = True  #@param {type:"boolean"}

## A1. 会话与 GPU

In [ ]:
#@title A1 · Session and GPU
import base64
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

SESSION_START = time.time()
DEADLINE = SESSION_START + 3600 * BUDGET_HOURS
REPO_DIR = "/content/RHO-FFT"
WORK_P1 = "/content/work/p1"         # results of the P1 pipeline: zeroth-order checks, controls (local disk, fast)
WORK_P2 = "/content/work/p2"         # results of the P2 pipeline: dose-response
P0_WORK = "/content/work/p0"         # results of the baseline regression (golden values, verdicts)
TINY = "/content/work/tiny"          # results of the CPU dry run, kept apart from the real ones
BUNDLE = "/content/work/session2_bundle.zip"
CACHE = "/content/cache"             # models, datasets, token caches and manifests
VENV = "/content/venv"


def sh(command, cwd=None, env=None, check=True):
    # Run a command, stream its output into the cell, and stop on failure when check is set.
    shown = command if isinstance(command, str) else " ".join(str(part) for part in command)
    print(f"$ {shown}", flush=True)
    process = subprocess.Popen(command, shell=isinstance(command, str), cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {shown}")
    return code


def hours_left():
    return (DEADLINE - time.time()) / 3600


STATUS = {}  # one line per stage, printed in the last cell


def release_gpu(delay=10):
    # Hand the GPU runtime back. Everything on the VM's local disk is lost, so this runs only after the results
    # have been shown and downloaded.
    if not RELEASE_GPU:
        print("RELEASE_GPU is off; the runtime stays connected.")
        return
    print(f"Releasing the GPU runtime in {delay} seconds ...", flush=True)
    time.sleep(delay)
    from google.colab import runtime
    runtime.unassign()


def stop(message):
    # A failed check stops "Run all"; the error handler below then releases the GPU.
    raise RuntimeError(message)


def build_bundle():
    # One zip with both pipelines' bundles (runs, logs, CSV files), their reports and figures. Returns its path or None.
    import zipfile
    parts = [Path(WORK_P1) / "feedback" / "feedback_P1_bundle.zip", Path(WORK_P2) / "feedback" / "feedback_P2_bundle.zip"]
    extras = [*Path(WORK_P1).glob("feedback/*.md"), *Path(WORK_P1).glob("feedback/*.png"), *Path(WORK_P2).glob("feedback/*.md"),
              *Path(WORK_P2).glob("feedback/*.png"), *Path(WORK_P1).glob("zo_check/zo_check.*")]
    present = [path for path in parts + extras if path.exists()]
    if not present:
        return None
    with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in present:
            bundle.write(path, f"{path.parent.parent.name}/{path.parent.name}/{path.name}")
    return BUNDLE


def download_results(refresh=True):
    # Send the result bundle to the browser. With refresh, the feedback is rebuilt first from whatever exists, so
    # that an interrupted session still leaves with its finished runs. Returns whether a download was started.
    if refresh and "p1" in globals():
        for runner, root in (("p1", WORK_P1), ("p2", WORK_P2)):
            if Path(root).exists():
                try:
                    globals()[runner]("feedback", extra=["--quiet"])
                except Exception as error:
                    print(f"{runner} feedback could not be rebuilt:", repr(error))
    bundle = build_bundle()
    if not (DOWNLOAD_BUNDLE and bundle):
        return False
    try:
        from google.colab import files
        files.download(bundle)
        return True
    except Exception as error:
        print("download failed:", repr(error))
        return False


def _release_on_error(shell, etype, value, traceback, tb_offset=None):
    # Any error in any later cell: show it, save what exists, then release the GPU instead of leaving it idle.
    shell.showtraceback((etype, value, traceback), tb_offset=tb_offset)
    download_results()
    release_gpu(30 if DOWNLOAD_BUNDLE else 10)


# Manual interruption (KeyboardInterrupt) is not an Exception and does not release the runtime.
get_ipython().set_custom_exc((Exception,), _release_on_error)


sh("nvidia-smi", check=False)

## A2. 取代码

克隆仓库并检出 `REPO_REF`。回归检查要用 `git archive` 取出两个参照提交（上游原样导入、插桩提交），所以需要带完整历史的克隆。token 只通过环境变量传给 git，不会出现在输出里。

In [ ]:
#@title A2 · Get the code
def git_environment():
    # GitHub credentials for a private repository, passed to git through the environment only.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as error:
        print("Colab secret GITHUB_TOKEN not available:", type(error).__name__)
    if not token and ASK_FOR_TOKEN:
        from getpass import getpass
        token = getpass("GitHub token with read access (leave empty for a public repository): ").strip()
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    return env


source, git_env = REPO_URL, git_environment()

if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh(["git", "clone", "--quiet", source, REPO_DIR], env=git_env)
else:
    sh(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], env=git_env)
remote_ref = f"origin/{REPO_REF}"
has_remote_ref = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", remote_ref],
                                capture_output=True).returncode == 0
sh(["git", "-C", REPO_DIR, "checkout", "--quiet", "--detach", remote_ref if has_remote_ref else REPO_REF])
sh(["git", "-C", REPO_DIR, "log", "--oneline", "-8"])

revisions = json.loads(Path(f"{REPO_DIR}/experiments/p0/revisions.json").read_text())
for name, sha in revisions.items():
    if subprocess.run(["git", "-C", REPO_DIR, "cat-file", "-e", sha + "^{commit}"], capture_output=True).returncode != 0:
        raise RuntimeError(f"Reference revision {name} ({sha}) is missing from this clone; a full-history clone is required.")
print("reference revisions present:", revisions)

## A3. 环境

建一个独立的 Python 3.11 虚拟环境，按 `pyproject.toml` 的固定版本安装依赖（PyTorch 2.6.0 / CUDA 12.4）。所有训练都在这个环境的子进程里运行。仓库本身不安装，运行时通过 `PYTHONPATH` 指向源码树。

G4 运行时是 Blackwell 架构的 RTX PRO 6000，固定版本的 PyTorch 2.6.0（CUDA 12.4）不支持它。环境建好后会先探测 GPU：固定版本用不了时，自动换成参数 `TORCH_FALLBACK` 指定的版本（CUDA 12.8 构建），并在结果里记录实际使用的版本。回归检查在同一环境里进行，所以基线是否被改坏的结论不受影响；与论文数值的对照则多了一层环境差异。H100 上直接使用固定版本。

In [ ]:
#@title A3 · Environment
def clean_environment():
    # Colab exports constraint files for its own package set (PIP_CONSTRAINT, UV_CONSTRAINT, ...). They must not
    # apply to the separate environment built here, which pins different versions.
    return {key: value for key, value in os.environ.items() if not key.startswith(("UV_", "PIP_"))}


if ENV_MODE == "venv":
    uv = f"{sys.executable} -m uv"
    sh(f"{sys.executable} -m pip install -q uv")
    if not os.path.exists(f"{VENV}/bin/python"):
        if sh(f"{uv} venv --python 3.11 {VENV}", env=clean_environment(), check=False) != 0:
            print("Python 3.11 could not be provisioned; building the environment on the runtime's own Python instead")
            shutil.rmtree(VENV, ignore_errors=True)
            sh(f"{uv} venv --python {sys.executable} {VENV}", env=clean_environment())
    install = f"{uv} pip install --python {VENV}/bin/python -r pyproject.toml --extra dev"
    if sh(install + " --torch-backend cu124", cwd=REPO_DIR, env=clean_environment(), check=False) != 0:
        print("retrying without --torch-backend (the default Linux wheel of torch 2.6.0 is the CUDA 12.4 build)")
        sh(install, cwd=REPO_DIR, env=clean_environment())
    PY = f"{VENV}/bin/python"
else:
    PY = sys.executable
    sh(f"{PY} -m pip install -q transformers==4.56.2 peft==0.17.1 datasets==3.6.0 accelerate==1.10.1 "
       "safetensors==0.6.2 sentencepiece==0.2.1 pytest", env=clean_environment())

sh([PY, "-c", "import torch, transformers, peft, datasets; "
              "print('torch', torch.__version__, '| cuda runtime', torch.version.cuda, '| cuda available', torch.cuda.is_available()); "
              "print('transformers', transformers.__version__, '| peft', peft.__version__, '| datasets', datasets.__version__)"])

# Probe the GPU with the interpreter that will train. The pinned PyTorch has no kernels for Blackwell GPUs (the G4
# runtime); if it cannot use this GPU, the fallback build for CUDA 12.8 is installed and probed again. The versions
# actually used are recorded with every result, and the regression check runs in the same environment.
GPU_PROBE = ("import torch; x = torch.randn(256, 256, device='cuda', dtype=torch.bfloat16); "
             "assert torch.isfinite((x @ x).float()).all(); "
             "print('GPU usable:', torch.cuda.get_device_name(0), '| torch', torch.__version__)")


def gpu_usable():
    return sh([PY, "-c", GPU_PROBE], check=False) == 0


if shutil.which("nvidia-smi") is None:
    stop("No GPU in this runtime. Choose a GPU runtime type and run again.")
if not gpu_usable():
    if ENV_MODE != "venv":
        stop("The runtime's own PyTorch cannot use this GPU. Send the output above back.")
    print(f"\nThe pinned PyTorch cannot use this GPU; installing torch {TORCH_FALLBACK} built for CUDA 12.8 instead.")
    fallback = f"{uv} pip install --python {VENV}/bin/python torch=={TORCH_FALLBACK}"
    if sh(fallback + " --torch-backend cu128", env=clean_environment(), check=False) != 0:
        sh(fallback + " --index-url https://download.pytorch.org/whl/cu128", env=clean_environment())
    if not gpu_usable():
        stop("PyTorch cannot use this GPU with either build. Send the output above back.")
    STATUS["environment note"] = f"torch {TORCH_FALLBACK} (CUDA 12.8) in place of the pinned 2.6.0, which does not support this GPU"


def p0(*stages, root=P0_WORK, extra=()):
    # Run stages of experiments/p0/p0.py (baseline regression) with the training interpreter; 0 = all stages ok.
    command = [PY, "experiments/p0/p0.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda", *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def p1(*stages, root=WORK_P1, extra=()):
    # Stages of experiments/p1/p1.py (queue of arms, feedback); 0 = all stages ok.
    command = [PY, "experiments/p1/p1.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda",
               "--model", MODEL, "--seeds", str(SEED), *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def p2(*stages, root=WORK_P2, extra=()):
    # Stages of experiments/p2/p2.py (base run, counterfactual dose-response, feedback); 0 = all stages ok.
    command = [PY, "experiments/p2/p2.py", *stages, "--root", root, "--cache-dir", CACHE, "--device", "cuda",
               "--model", MODEL, "--seed", str(SEED), *extra]
    return sh(command, cwd=REPO_DIR, check=False)


def show_feedback(which=("p1", "p2"), download=True):
    # Rebuild the feedback from the runs finished so far, show it, and send the bundle to the browser.
    from IPython.display import Markdown, display
    for name in which:
        runner, root, report = {"p1": (p1, WORK_P1, "feedback_P1.md"), "p2": (p2, WORK_P2, "feedback_P2.md")}[name]
        runner("feedback", extra=["--quiet"])
        path = Path(root) / "feedback" / report
        if path.exists():
            display(Markdown(path.read_text()))
    if download:
        started = download_results(refresh=False)
        print("result bundle:", "download started" if started else "not downloaded")


def allowed(label):
    # Whether a GPU stage may start: the regression must have passed (or FORCE) and some budget must be left.
    if regression_ok is False and not FORCE:
        print(f"{label} skipped: the regression check did not pass. Set FORCE = True to run it anyway.")
        STATUS[label] = "skipped (regression did not pass)"
        return False
    if hours_left() < 0.3:
        print(f"{label} skipped: less than 18 minutes of budget left.")
        STATUS[label] = "skipped (no budget left)"
        return False
    if regression_ok is None:
        print("NOTE: the regression check was not run in this session; the results build on an unverified baseline.")
    return True


def run_p1_block(block, label):
    # One block of arms.json under the remaining budget (ten minutes are kept for feedback and download).
    if not allowed(label):
        return False
    done = p1("queue", extra=["--blocks", str(block), "--jobs", str(JOBS), "--deadline", str(DEADLINE - 600)]) == 0
    STATUS[label] = "all runs finished" if done else "partly done (see the queue table in the report)"
    print(f"{hours_left():.2f} h of budget left")
    show_feedback(("p1",))
    return done


def run_p2_stage(stage, label, show=True):
    if not allowed(label):
        return False
    done = p2(stage, extra=["--jobs", str(JOBS), "--deadline", str(DEADLINE - 600)]) == 0
    STATUS[label] = "finished" if done else "not finished (see the report)"
    print(f"{hours_left():.2f} h of budget left")
    if show:
        show_feedback(("p2",))
    return done

## B. 预演（CPU，约四分钟）

用随机初始化的小模型和合成数据，把基线回归、剂量响应（起点运行加反事实轨迹）、零阶估计器检查和 P1 的检验臂各走一遍。只检查流程能不能跑通，数字没有意义。SQuAD 的检验臂不在预演里（小模型的分词器不支持生成式评估）。失败时 notebook 停下并释放 GPU。

In [ ]:
#@title B · Dry run on CPU
if RUN_TINY_DRY_RUN:
    steps = [("baseline regression", [PY, "experiments/p0/p0.py", "env", "golden", "regress", "--tiny", "--root", f"{TINY}/p0"]),
             ("dose-response", [PY, "experiments/p2/p2.py", "all", "--tiny", "--quiet", "--jobs", "2", "--root", f"{TINY}/p2"])]
    for label, command in steps:
        code = sh(command, cwd=REPO_DIR, check=False)
        STATUS[f"dry run: {label}"] = "ok" if code == 0 else "FAILED"
        if code != 0:
            stop(f"The dry run of the {label} failed. Nothing has been run on the GPU yet; send the output above back.")
    tiny_state = next(Path(TINY).glob("p2/runs/*dose-base/states/*.safetensors"))
    steps = [("estimator check", [PY, "-m", "ho_fl.zo_check", "--tiny", "--states", str(tiny_state), "--q", "16", "--batches", "1",
                                  "--output", f"{TINY}/p1/zo_check"]),
             ("P1 checks and controls", [PY, "experiments/p1/p1.py", "queue", "feedback", "--tiny", "--quiet", "--jobs", "2",
                                         "--blocks", "5,4", "--root", f"{TINY}/p1"])]
    for label, command in steps:
        code = sh(command, cwd=REPO_DIR, check=False)
        STATUS[f"dry run: {label}"] = "ok" if code == 0 else "FAILED"
        if code != 0:
            stop(f"The dry run of the {label} failed. Nothing has been run on the GPU yet; send the output above back.")
    print(f"\ndry run ok, {hours_left():.2f} h of budget left")

## C1. 环境记录与单元测试

记录 GPU 型号、驱动和各依赖的版本，并做一次 BF16 探测；在固定版本的环境里跑一遍不需要 GPU 的测试（约一分钟）。

In [ ]:
#@title C1 · Environment record and unit tests
env_ok = p0("env") == 0
STATUS["environment"] = "ok" if env_ok else "FAILED"
if not env_ok:
    stop("Environment check failed (no CUDA device, bfloat16 not usable, or a reference revision is missing). See the warnings above.")
if RUN_TESTS:
    tests_ok = p0("tests") == 0
    STATUS["unit tests"] = "ok" if tests_ok else "FAILURES"
    print("unit tests:", "ok" if tests_ok else "FAILURES (listed above)")

## C2. 黄金值与基线回归（I1、I2）

三个冒烟配置（HO-FL、FedAvg、DeComFL，OPT-125M / SST-2，3 轮）在上游原样提交上各跑一次，在插桩提交上各跑两次，在当前代码上各跑一次，再加两次 `ho-fl-replay`。当前代码改动了 `hybrid_step` 和 `local_step`（新增默认关闭的关键字参数），这一步检查默认配置下的结果仍与上游一致。

读法：`I1 verdict` 为 `PASS_BITWISE` 或 `PASS_TOLERANCE` 即通过；`FAIL` 说明当前代码改变了基线，后面的实验组默认不再运行；`INCONCLUSIVE` 说明这块 GPU 上参照提交自己两次运行的差就超过了容差，需要研究负责人决定怎么判。

In [ ]:
#@title C2 · Golden values and regression (I1, I2)
regression_ok = None
if RUN_REGRESSION:
    golden_ok = p0("golden") == 0
    regression_ok = golden_ok and p0("regress") == 0
    STATUS["golden values"] = "ok" if golden_ok else "ATTENTION"
    STATUS["regression (I1, I2)"] = "ok" if regression_ok else "ATTENTION"
    p0("feedback", extra=["--quiet"])
    # The regression record travels with both bundles.
    for work in (WORK_P1, WORK_P2):
        target = Path(work) / "p0_feedback"
        target.mkdir(parents=True, exist_ok=True)
        for path in [*Path(P0_WORK).glob("feedback/feedback_P0.*"), *Path(P0_WORK).glob("golden/*/golden.json"),
                     *Path(P0_WORK).glob("regress/*/verdicts.json"), Path(P0_WORK) / "env.json"]:
            if path.exists():
                shutil.copy2(path, target / (path.parent.name + "__" + path.name if path.name in ("golden.json", "verdicts.json") else path.name))
    print("\ngolden:", "ok" if golden_ok else "ATTENTION", "| regression (I1, I2):", "ok" if regression_ok else "ATTENTION")
    print(f"{hours_left():.2f} h of budget left")

## C3. 零阶检验，AG News（3 次运行，约 35 分钟）

- `A-b0.75-attr`：HO-FL 默认设置（β=0.75），加归因探针。每 4 轮在 1,024 条固定的评估样本上求一次梯度，把这一轮实际施加的更新带来的（一阶近似的）损失下降，拆成一阶分量和零阶分量各自的贡献，按层段汇总。探针只读，训练与第一次会话的 `A-b0.75` 完全相同：报告会逐个评估点比对，相同则两次会话的结果可以配对。
- `Au24-b0`：所有设备都用最高的零阶分界（第 24 层，只有最上面 4 层做一阶），β=0，复现论文表 9 的 63.22；同样加归因探针。
- `Bu24-b0`：同一设置下冻结底层（零阶关闭，直接平均），即这个设置下的 Top-FO-Only。

AG News 上默认设置的 Top-FO-Only（作者 83.37 对 86.69）用第一次会话的 `B-b0.75` 和 `A-b0.75`（85.95 对 85.99），不重跑。

In [ ]:
#@title C3 · Zeroth-order check on AG News (block 5)
if RUN_ZO_AG_NEWS:
    run_p1_block(5, "zeroth-order check, AG News")

## C4. 剂量响应：起点运行与反事实轨迹（约 15 分钟）

先用默认设置跑 40 轮，保存第 40 轮的状态作为起点；再从这个状态跑一条 20 轮的轨迹（名单按剂量 m=2 的布局：指定的那台 T1 设备加上另外 5 台）。每一轮，另外 5 台设备都在另一个分界上用同样的批次再训练一次，于是同一轮里可以组合出 m=0 到 5 的所有剂量；再对每个剂量用 8 种聚合规则、3 种故障强度（无、取反、取反并放大 20 倍）各算一次聚合结果，记录故障的影响和损失的下降。探针只读，不改变这条轨迹。

In [ ]:
#@title C4 · Dose-response: base run and counterfactual trajectory
if RUN_DOSE_CF:
    if run_p2_stage("base", "dose-response base run", show=False):
        run_p2_stage("cf", "counterfactual dose-response")
    else:
        STATUS["counterfactual dose-response"] = "skipped (no base run)"

## C5. 零阶估计器的数值检查（不训练，约 10 分钟）

在初始状态和上一步保存的第 40 轮状态上，对三档设备各取 2 个批次、64 个方向：调用训练用的 `hybrid_step` 得到 HO-FL 的标量 v，再用 autograd 算出同一方向上的精确方向导数，比较 v/μ 与它（μ 取 1e-4、1e-3、1e-2）。先在 BF16 主干上做（与训练相同），再把主干转成 FP32 做一遍作为参照。另外给出 q 个方向的估计与真实梯度的余弦随 q 的变化，以及理论值 √(q/d)。

In [ ]:
#@title C5 · Numerical check of the zeroth-order estimator
if RUN_ZO_CHECK and allowed("zeroth-order estimator check"):
    base_state = Path(WORK_P2) / "runs" / f"{MODEL}-s{SEED}-dose-base" / "states" / "round_0040.safetensors"
    states = str(base_state) if base_state.exists() else ""
    code = sh([PY, "-m", "ho_fl.zo_check", "--states", states, "--output", f"{WORK_P1}/zo_check", "--model", MODEL, "--dataset", "ag_news",
               "--partition", "dirichlet_05", "--seed", str(SEED), "--device", "cuda", "--cache-dir", CACHE], cwd=REPO_DIR, check=False)
    STATUS["zeroth-order estimator check"] = "ok" if code == 0 else "FAILED (see above)"
    report = Path(WORK_P1) / "zo_check" / "zo_check.md"
    if report.exists():
        from IPython.display import Markdown, display
        display(Markdown(report.read_text()))
    print(f"{hours_left():.2f} h of budget left")

## C6. 零阶检验，SQuAD：Top-FO-Only（2 次运行，可选，最贵）

作者在 rebuttal 里报告 SQuAD 上冻结底层后 F1 从 86.92 降到 83.65。这里按同样的联邦设置跑 HO-FL（β=0.75）和冻结底层两次，只在最后做一次完整验证集的生成式评估（EM/F1），不做中间评估。先跑一次 3 轮的预检来测耗时；放不进剩余预算会被跳过。

In [ ]:
#@title C6 · Zeroth-order check on SQuAD (block 6)
if RUN_ZO_SQUAD:
    run_p1_block(6, "zeroth-order check, SQuAD")

## C7. 对照（4 次运行，可选，约 1.2 小时）

`Cx2-b0`（`fo_mean` 步长 ×2，对比第一次会话的 `Ax2-b0`：步长相同时只让一阶报告投票是否仍更好）、`fedavg-x2`（上限本身是否欠步长）、`C-b0-r480` 和 `A-b0-r480`（训练三倍长，收敛后差距是否还在）。这些运行偏离"不单独调参"的规则，只作诊断。

In [ ]:
#@title C7 · Controls (block 4)
if RUN_CONTROLS:
    run_p1_block(4, "controls")

## D. 两份反馈报告与图

P1 报告的第 2 节是零阶检验：作者的表述、作者的数字、我们的数字和判定，后面是归因表和估计器检查表。P2 报告的第 3 节是反事实剂量响应：每种故障强度下各规则在各剂量的进展与影响，以及 H2 的判定和"一阶报告不足时哪种回退最好"。

In [ ]:
#@title D · Feedback reports and figures
from IPython.display import Image, display

show_feedback(download=False)
STATUS["feedback"] = "written"


def read_feedback(root, name):
    path = Path(root) / "feedback" / name
    return json.loads(path.read_text()) if path.exists() else {}


data_p1, data_p2 = read_feedback(WORK_P1, "feedback_P1.json"), read_feedback(WORK_P2, "feedback_P2.json")
# The figures are drawn by the notebook's own Python, which has matplotlib.
for script, root, name in (("p1/plot_curves.py", WORK_P1, "feedback_P1.json"), ("p2/plot_dose.py", WORK_P2, "feedback_P2.json")):
    if (Path(root) / "feedback" / name).exists():
        sh([sys.executable, f"{REPO_DIR}/experiments/{script}", f"{root}/feedback/{name}", f"{root}/feedback"], check=False)
for path in [Path(WORK_P1) / "feedback" / name for name in ("p1_zo_check.png", "p1_curves.png", "p1_controls.png")] + [Path(WORK_P2) / "feedback" / "p2_dose_cf.png"]:
    if path.exists():
        display(Image(filename=str(path)))
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h")

## E. 汇总并释放 GPU

打印各步骤的状态和主要判定，让浏览器下载结果包，然后释放 GPU 运行时。

In [ ]:
#@title E · Summary, download, then release the GPU
data_p1, data_p2 = globals().get("data_p1") or {}, globals().get("data_p2") or {}
print("stage status")
for name, state in STATUS.items():
    print(f"  {name:36s} {state}")
for statement in data_p1.get("statements", []):
    print(f"  ZO statement: {statement['source']} | {statement['task']}: {statement['verdict']}")
if data_p2.get("cf"):
    print("  H2 (counterfactual):", ", ".join(f"{fault} {entry['verdict']}" for fault, entry in data_p2["cf"]["h2"].items()))
print(f"session time used: {(time.time() - SESSION_START) / 3600:.2f} h of {BUDGET_HOURS} h")
started = download_results(refresh=True)
print("result bundle:", "download started" if started else "NOT downloaded; the reports above are the record of this session")
release_gpu(30 if started else 10)

## 带回代码库的内容

- 页面上显示的两份报告：**`feedback_P1.md`**（第 2 节：零阶检验）和 **`feedback_P2.md`**（第 3 节：反事实剂量响应）。
- 浏览器下载的 **`session2_bundle.zip`**：两份报告与图，以及两个流水线各自的结果包（每个运行的 `config.json`、`summary.json`、`curves.jsonl`、`ledger.jsonl`、`attribution.jsonl`、`dose_probe.jsonl`、估计器检查的结果、计时和日志、回归记录）。
- 把整页输出存成 `colab_results_feedback/v2_1122/` 下的文件，和第一次会话一样。